# Objective 1 - Build the N=2 / N>2 dataset
Sources (all public, no API key needed):
1. **DeFiHackLabs** (SunWeb3Sec, Apache-2.0): Foundry PoCs of real DeFi exploits. Gives incident, date, vulnerability type, chain, pre-attack fork block and every on-chain address the exploit touches. https://github.com/SunWeb3Sec/DeFiHackLabs
2. **On-chain bytecode** at the pre-attack block via free public archive RPCs (`eth_getCode`): separates contracts from wallets (gives N) and stores runtime bytecode for CFG building.

Run from the `objective1/` folder. Outputs are already built in `data/processed/`; re-run only to refresh.

In [1]:
import re, json, csv, time, hashlib, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd, requests
DATA = Path.cwd() / "data"
(DATA / "raw").mkdir(parents=True, exist_ok=True)

## 1. Download DeFiHackLabs (skipped if present)

In [ ]:
REPOS = {"DeFiHackLabs": "https://github.com/SunWeb3Sec/DeFiHackLabs.git"}
for name, url in REPOS.items():
    dst = DATA / "raw" / name
    if not dst.exists():   # core.longpaths avoids Windows path-length checkout failures
        subprocess.run(["git", "-c", "core.longpaths=true", "clone", "--depth", "1", url, str(dst)], check=True)
    print(name, subprocess.run(["git", "-C", str(dst), "log", "-1", "--format=%H %cd"], capture_output=True, text=True).stdout.strip())

## 2. Parse DeFiHackLabs into an incident index
Labels are keyword heuristics on the README vulnerability text (`label_access_control`, `label_flash_loan_type`) and on PoC code (`uses_flash_loan`). Review `vuln_type` before trusting them.

In [3]:
REPO = DATA / "raw" / "DeFiHackLabs"
OUT = DATA / "processed"

HEAD = re.compile(r"^###\s+(\d{8})\s*-?\s*(\S.*?)\s+-\s+(.+?)\s*$")
LINK = re.compile(r"\]\((?:\.\./\.\./)?(src/test/[^)]+?\.sol)\)")
LOST = re.compile(r"^###\s*Lost:\s*(.+)$", re.I)

entries = {}
for md in [REPO / "README.md", *sorted((REPO / "past").glob("*/README.md"))]:
    cur = None
    for line in md.read_text(encoding="utf-8", errors="ignore").splitlines():
        m = HEAD.match(line)
        if m:
            cur = dict(date=m[1], name=m[2], vuln_type=m[3], lost="", readme=md.relative_to(REPO).as_posix())
            continue
        if cur is None:
            continue
        lm = LOST.match(line)
        if lm and not cur["lost"]:
            cur["lost"] = lm[1][:200]
        k = LINK.search(line)
        if k and (REPO / k[1]).exists() and k[1] not in entries:
            entries[k[1]] = dict(cur, poc=k[1])
            cur = None

FORK = re.compile(r'createSelectFork\(\s*"(\w+)"\s*,\s*([^)]+)\)')
ADDR = re.compile(r"0x[0-9a-fA-F]{40}\b")
TXH = re.compile(r"0x[0-9a-fA-F]{64}")
FLASH = re.compile(r"flash\s*loan|flashloan|flashLoan|\.flash\(|DPPFlashLoanCall|pancakeCall|uniswapV2Call|uniswapV3FlashCallback|receiveFlashLoan|executeOperation|onFlashLoan", re.I)
AC_TYPE = re.compile(r"access control|unprotected|lack of access|missing access|unauthori|no access|permission|arbitrary (external )?call|arbitrary.*transfer|unverified|privilege|onlyowner|missing check on caller", re.I)
FL_TYPE = re.compile(r"flash", re.I)

def resolve_block(expr, code, depth=0):
    expr = expr.replace("_", "").strip()
    terms = re.findall(r"([+-]?)\s*(\w+)", expr)
    if not terms or depth > 3: return ""
    total = 0
    for sign, t in terms:
        if not t.isdigit():
            mm = re.search(rf"\b{t}\s*=\s*([^;]+);", code)
            v = resolve_block(mm[1], code, depth + 1) if mm else ""
            if not v: return ""
            t = v
        total += -int(t) if sign == "-" else int(t)
    return str(total)


rows = []
for poc, e in entries.items():
    src = (REPO / poc).read_text(encoding="utf-8", errors="ignore")
    code = re.sub(r"//.*", "", src)                       # drop comments (analysis links, tx urls)
    f = FORK.search(code)
    chain, block = (f[1], f[2].strip()) if f else ("", "")
    block = resolve_block(block, code)
    addrs = sorted({a.lower() for a in ADDR.findall(code)} - {"0x" + "0" * 40})
    txs = TXH.findall(src)
    rows.append(dict(
        incident_id=Path(poc).stem, date=e["date"], name=e["name"], vuln_type=e["vuln_type"], lost=e["lost"],
        chain=chain, fork_block=block, poc=poc, attack_tx=txs[0] if txs else "",
        n_candidate_addresses=len(addrs), addresses=";".join(addrs),
        label_access_control=int(bool(AC_TYPE.search(e["vuln_type"]))),
        label_flash_loan_type=int(bool(FL_TYPE.search(e["vuln_type"]))),
        uses_flash_loan=int(bool(FLASH.search(code))),
    ))

df = pd.DataFrame(rows).sort_values("date")
# the same PoC filename can appear in two month folders -> suffix the month to keep ids unique
dup = df.incident_id.duplicated(keep=False)
df.loc[dup, "incident_id"] = df.loc[dup, "incident_id"] + "_" + df.loc[dup, "poc"].str.extract(r"src/test/([^/]+)/")[0]
OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT / "dhl_incidents.csv", index=False)
print(len(entries), "readme entries linked to PoCs;", len(df), "rows")
print(df.chain.value_counts().head(8).to_dict())
print("no fork block:", (df.fork_block == "").sum())
print("access_control:", df.label_access_control.sum(), "| flash in type:", df.label_flash_loan_type.sum(), "| uses flash loan:", df.uses_flash_loan.sum())
print(df.vuln_type.str.lower().value_counts().head(25).to_string())
print(df.n_candidate_addresses.describe().to_string())

869 readme entries linked to PoCs; 869 rows
{'bsc': 370, 'mainnet': 336, 'arbitrum': 38, 'base': 38, 'polygon': 26, '': 19, 'avalanche': 13, 'optimism': 12}
no fork block: 85
access_control: 142 | flash in type: 58 | uses flash loan: 472
vuln_type
business logic flaw                      110
price manipulation                        57
access control                            50
logic flaw                                37
reentrancy                                32
lack of access control                    16
flashloan price manipulation              13
insufficient validation                   11
arbitrary external call vulnerability     10
price oracle manipulation                  9
reflection token                           9
arbitrary external call                    9
flashloan attack                           7
flashloan & price oracle manipulation      7
arbitrary call                             7
incorrect access control                   7
precision loss                  

## 3. Fetch bytecode at the pre-attack block
Resumable cache in `data/raw/bytecode/<chain>/index.csv` plus one `.hex` per unique code. These free endpoints served historical state on 2026-10-12; swap one if it starts failing.

In [4]:
def fetch_chain(CHAIN, RPC, WORKERS=3):
    BASE = DATA
    OUTD = BASE / "raw" / "bytecode" / CHAIN
    OUTD.mkdir(parents=True, exist_ok=True)
    IDX = OUTD / "index.csv"

    df = pd.read_csv(BASE / "processed" / "dhl_incidents.csv", dtype=str).fillna("")
    df = df[(df.chain == CHAIN) & (df.fork_block != "")]
    jobs = sorted({(a, int(r.fork_block)) for r in df.itertuples() for a in r.addresses.split(";") if a})

    done = set()
    if IDX.exists():
        for r in csv.DictReader(open(IDX)):
            if r["status"] == "ok": done.add((r["address"], int(r["block"])))
    jobs = [j for j in jobs if j not in done]
    print(f"{CHAIN}: {len(df)} incidents, {len(jobs)} (address, block) lookups to do, {len(done)} cached", flush=True)

    sess = requests.Session()


    def get_code(job):
        addr, block = job
        payload = {"jsonrpc": "2.0", "id": 1, "method": "eth_getCode", "params": [addr, hex(block)]}
        for attempt in range(6):
            try:
                r = sess.post(RPC, json=payload, timeout=30).json()
                if "result" in r:
                    return addr, block, "ok", r["result"]
                msg = str(r.get("error", r))[:120]
            except Exception as e:
                msg = str(e)[:120]
            time.sleep(2 * (attempt + 1))
        return addr, block, "error", msg


    new = not IDX.exists()
    with open(IDX, "a", newline="") as f, ThreadPoolExecutor(WORKERS) as ex:
        w = csv.writer(f)
        if new: w.writerow(["address", "block", "status", "is_contract", "code_sha", "code_len", "error"])
        for i, (addr, block, status, res) in enumerate(ex.map(get_code, jobs), 1):
            if status == "ok":
                code = res[2:] if res.startswith("0x") else res
                sha = hashlib.sha256(code.encode()).hexdigest()[:16] if code else ""
                if code and not (OUTD / f"{sha}.hex").exists(): (OUTD / f"{sha}.hex").write_text(code)
                w.writerow([addr, block, "ok", int(bool(code)), sha, len(code) // 2, ""])
            else:
                w.writerow([addr, block, "error", "", "", "", res])
            if i % 200 == 0: f.flush(); print(f"  {i}/{len(jobs)}", flush=True)
    print("done", flush=True)

In [5]:
RPCS = {"mainnet": "https://eth.drpc.org", "bsc": "https://bsc-mainnet.public.blastapi.io",
        "arbitrum": "https://arbitrum-one.public.blastapi.io", "base": "https://base.drpc.org",
        "polygon": "https://polygon.drpc.org", "optimism": "https://optimism.drpc.org",
        "avalanche": "https://api.avax.network/ext/bc/C/rpc"}
for chain, rpc in RPCS.items():
    fetch_chain(chain, rpc)

mainnet: 303 incidents, 0 (address, block) lookups to do, 2661 cached


done


bsc: 354 incidents, 0 (address, block) lookups to do, 2621 cached


done


arbitrum: 33 incidents, 0 (address, block) lookups to do, 334 cached


...

done


optimism: 12 incidents, 0 (address, block) lookups to do, 109 cached


done


avalanche: 12 incidents, 0 (address, block) lookups to do, 102 cached


done


## 4. Contract counts and N buckets
`n_contracts` = protocol contracts with code + 1 attacker. Shared infrastructure (addresses used by 5+ incidents on a chain: WETH, USDT, routers) is excluded; `n_contracts_total` keeps it.

In [ ]:
D = DATA
inc = pd.read_csv(D / "processed" / "dhl_incidents.csv", dtype=str).fillna("")

# --- bytecode lookups (prefer an ok row over an earlier error row) ---
idx = []
for f in (D / "raw" / "bytecode").glob("*/index.csv"):
    t = pd.read_csv(f, dtype=str).fillna(""); t["chain"] = f.parent.name; idx.append(t)
idx = pd.concat(idx).sort_values("status", ascending=False).drop_duplicates(["chain", "address", "block"], keep="first")
idx = idx[idx.status == "ok"]
key = {(r.chain, r.address, r.block): (r.is_contract == "1", r.code_sha) for r in idx.itertuples()}

# shared infrastructure = contract addresses used by >= 5 different incidents on the same chain (WETH, USDT, routers, ...)
uses = {}
for r in inc.itertuples():
    for a in filter(None, r.addresses.split(";")):
        uses.setdefault((r.chain, a), set()).add(r.incident_id)
shared = {k for k, v in uses.items() if len(v) >= 5}

BUCKETS = [(2, 2, "2"), (3, 5, "3-5"), (6, 10, "6-10"), (11, 10**9, "11+")]
def bucket(n):
    for lo, hi, name in BUCKETS:
        if lo <= n <= hi: return name
    return "1"

rows = []
for r in inc.itertuples():
    rec = r._asdict(); rec.pop("Index")
    addrs = [a for a in r.addresses.split(";") if a]
    looked = [(a, key.get((r.chain, a, r.fork_block))) for a in addrs]
    rec["bytecode_available"] = int(bool(r.fork_block) and all(v is not None for _, v in looked) and bool(addrs))
    cons = [a for a, v in looked if v and v[0]]
    proto = [a for a in cons if (r.chain, a) not in shared]
    rec["contracts"] = ";".join(cons)
    rec["code_shas"] = ";".join(key[(r.chain, a, r.fork_block)][1] for a in cons)
    rec["n_contracts_total"] = len(cons) + 1          # + attacker contract (the PoC)
    rec["n_contracts_protocol"] = len(proto) + 1      # excluding shared infra (tokens/routers used by >=5 incidents)
    rec["n_contracts"] = rec["n_contracts_protocol"]  # primary N used for buckets
    rec["bucket"] = bucket(rec["n_contracts"]) if rec["bytecode_available"] else ""
    rows.append(rec)
out = pd.DataFrame(rows)
out.to_csv(D / "processed" / "dhl_incidents.csv", index=False)

ok = out[out.bytecode_available == 1]
print(f"incidents: {len(out)} | with full bytecode: {len(ok)}")
print("bucket counts (N = protocol contracts + attacker):")
print(pd.crosstab(ok.bucket, [ok.label_access_control.rename("AC"), ok.label_flash_loan_type.rename("FL_type")], margins=True).to_string())
print("by uses_flash_loan:", ok.groupby("bucket").uses_flash_loan.apply(lambda s: (s == "1").sum()).to_dict())
print("total-N buckets:", ok.n_contracts_total.map(bucket).value_counts().to_dict())


## 5. EVM helpers: disassembler, symbolic-stack path search, DeFiTail-style stitching
A **data path** = opcodes of one feasible, successfully-terminating execution of one public function. Feasibility is DeFiTail's
stack-height check: every jump target must resolve from the symbolic stack and the stack must never underflow. The calldata
selector is a constant, so the dispatcher resolves to exactly that function; unknown branches are explored both ways
(bounded DFS: 2 paths per function, each block visited at most 32 times per path, 8,000 instructions max).
**Stitching**: at each CALL / CALLCODE / DELEGATECALL / STATICCALL, the most recent pushed 4-byte constant that the callee
implements is taken as the called function, and the callee's function path is spliced in (signature matching, like DeFiTail).

In [7]:
from functools import lru_cache
from collections import defaultdict
import sys, solcx


M = (1 << 256) - 1
NAMES, EFFECT = {}, {}   # opcode -> name ; opcode -> (pops, pushes)
def _op(code, name, pops, pushes): NAMES[code] = name; EFFECT[code] = (pops, pushes)
for c, n, p, q in [
    (0x00, "STOP", 0, 0), (0x01, "ADD", 2, 1), (0x02, "MUL", 2, 1), (0x03, "SUB", 2, 1), (0x04, "DIV", 2, 1),
    (0x05, "SDIV", 2, 1), (0x06, "MOD", 2, 1), (0x07, "SMOD", 2, 1), (0x08, "ADDMOD", 3, 1), (0x09, "MULMOD", 3, 1),
    (0x0a, "EXP", 2, 1), (0x0b, "SIGNEXTEND", 2, 1), (0x10, "LT", 2, 1), (0x11, "GT", 2, 1), (0x12, "SLT", 2, 1),
    (0x13, "SGT", 2, 1), (0x14, "EQ", 2, 1), (0x15, "ISZERO", 1, 1), (0x16, "AND", 2, 1), (0x17, "OR", 2, 1),
    (0x18, "XOR", 2, 1), (0x19, "NOT", 1, 1), (0x1a, "BYTE", 2, 1), (0x1b, "SHL", 2, 1), (0x1c, "SHR", 2, 1),
    (0x1d, "SAR", 2, 1), (0x20, "KECCAK256", 2, 1), (0x30, "ADDRESS", 0, 1), (0x31, "BALANCE", 1, 1),
    (0x32, "ORIGIN", 0, 1), (0x33, "CALLER", 0, 1), (0x34, "CALLVALUE", 0, 1), (0x35, "CALLDATALOAD", 1, 1),
    (0x36, "CALLDATASIZE", 0, 1), (0x37, "CALLDATACOPY", 3, 0), (0x38, "CODESIZE", 0, 1), (0x39, "CODECOPY", 3, 0),
    (0x3a, "GASPRICE", 0, 1), (0x3b, "EXTCODESIZE", 1, 1), (0x3c, "EXTCODECOPY", 4, 0),
    (0x3d, "RETURNDATASIZE", 0, 1), (0x3e, "RETURNDATACOPY", 3, 0), (0x3f, "EXTCODEHASH", 1, 1),
    (0x40, "BLOCKHASH", 1, 1), (0x41, "COINBASE", 0, 1), (0x42, "TIMESTAMP", 0, 1), (0x43, "NUMBER", 0, 1),
    (0x44, "PREVRANDAO", 0, 1), (0x45, "GASLIMIT", 0, 1), (0x46, "CHAINID", 0, 1), (0x47, "SELFBALANCE", 0, 1),
    (0x48, "BASEFEE", 0, 1), (0x49, "BLOBHASH", 1, 1), (0x4a, "BLOBBASEFEE", 0, 1), (0x50, "POP", 1, 0),
    (0x51, "MLOAD", 1, 1), (0x52, "MSTORE", 2, 0), (0x53, "MSTORE8", 2, 0), (0x54, "SLOAD", 1, 1),
    (0x55, "SSTORE", 2, 0), (0x56, "JUMP", 1, 0), (0x57, "JUMPI", 2, 0), (0x58, "PC", 0, 1), (0x59, "MSIZE", 0, 1),
    (0x5a, "GAS", 0, 1), (0x5b, "JUMPDEST", 0, 0), (0x5c, "TLOAD", 1, 1), (0x5d, "TSTORE", 2, 0),
    (0x5e, "MCOPY", 3, 0), (0x5f, "PUSH0", 0, 1), (0xf0, "CREATE", 3, 1), (0xf1, "CALL", 7, 1),
    (0xf2, "CALLCODE", 7, 1), (0xf3, "RETURN", 2, 0), (0xf4, "DELEGATECALL", 6, 1), (0xf5, "CREATE2", 4, 1),
    (0xfa, "STATICCALL", 6, 1), (0xfd, "REVERT", 2, 0), (0xfe, "INVALID", 0, 0), (0xff, "SELFDESTRUCT", 1, 0)]:
    _op(c, n, p, q)
for i in range(1, 33): _op(0x5f + i, f"PUSH{i}", 0, 1)
for i in range(1, 17): NAMES[0x7f + i] = f"DUP{i}"; NAMES[0x8f + i] = f"SWAP{i}"
for i in range(5): _op(0xa0 + i, f"LOG{i}", 2 + i, 0)

SUCCESS = {0x00, 0xf3, 0xff}          # STOP, RETURN, SELFDESTRUCT
FAIL = {0xfd, 0xfe}                   # REVERT, INVALID
CALLS = {0xf1, 0xf2, 0xf4, 0xfa}      # CALL, CALLCODE, DELEGATECALL, STATICCALL
CALLDATASIZE = 4 + 32 * 16            # assume well-formed calldata so ABI length checks pass


def disassemble(hexcode):
    b = bytes.fromhex(hexcode[2:] if hexcode.startswith("0x") else hexcode)
    ins, i = [], 0
    while i < len(b):
        op = b[i]
        n = op - 0x5f if 0x60 <= op <= 0x7f else 0
        val = int.from_bytes(b[i + 1:i + 1 + n], "big") if n else (0 if op == 0x5f else None)
        ins.append((i, op, val))
        i += 1 + n
    return ins


class Contract:
    def __init__(self, hexcode, name=""):
        self.name = name
        self.ins = disassemble(hexcode)
        self.at = {pc: k for k, (pc, _, _) in enumerate(self.ins)}            # pc -> instruction index
        self.jumpdests = {pc for pc, op, _ in self.ins if op == 0x5b}
        self.selectors = self._dispatcher()

    def _dispatcher(self):
        """Selectors from the `PUSH4 sel ... EQ ... PUSHn dest JUMPI` dispatcher pattern."""
        sels, ins = set(), self.ins
        for k, (pc, op, v) in enumerate(ins):
            if op == 0x63 and v:
                window = [o for _, o, _ in ins[k + 1:k + 5]]
                if 0x14 in window and 0x57 in window:
                    sels.add(v)
        return sels


def _alu(op, a):
    """Constant-fold a few ops used by dispatchers / jump-target arithmetic; None = unknown."""
    if any(x is None for x in a):
        return None
    try:
        if op == 0x01: return (a[0] + a[1]) & M
        if op == 0x02: return (a[0] * a[1]) & M
        if op == 0x03: return (a[0] - a[1]) & M
        if op == 0x04: return a[0] // a[1] if a[1] else 0
        if op == 0x10: return int(a[0] < a[1])
        if op == 0x11: return int(a[0] > a[1])
        if op == 0x14: return int(a[0] == a[1])
        if op == 0x15: return int(a[0] == 0)
        if op == 0x16: return a[0] & a[1]
        if op == 0x17: return a[0] | a[1]
        if op == 0x18: return a[0] ^ a[1]
        if op == 0x19: return (~a[0]) & M
        if op == 0x1b: return (a[1] << a[0]) & M if a[0] < 256 else 0
        if op == 0x1c: return a[1] >> a[0] if a[0] < 256 else 0
    except Exception:
        return None
    return None


def function_paths(c, selector, max_paths=2, max_len=8000, max_steps=300000, max_visits=32, unroll=4):
    """Feasible successful execution paths of `selector` in contract c.
    Bounded loop unrolling: once the current block has been entered more than `unroll` times on this path, its
    JUMPI condition is treated as unknown, so constant-count loops can also take their exit branch.
    Returns list of paths; each path = list of (opcode, PUSH4 value or None)."""
    out, steps = [], 0
    # explicit DFS stack of (instruction index, symbolic stack, path, visit counts)
    todo = [(0, (), (), {}, None)]
    while todo and len(out) < max_paths and steps < max_steps:
        k, st, path, visits, last = todo.pop()
        st, path, visits = list(st), list(path), dict(visits)
        while True:
            steps += 1
            if k >= len(c.ins) or len(path) > max_len or steps > max_steps:
                break
            pc, op, v = c.ins[k]
            if op == 0x5b:
                visits[pc] = visits.get(pc, 0) + 1
                last = pc
                if visits[pc] > max_visits:
                    break
            path.append((op, v if op == 0x63 else None))
            if op in SUCCESS:
                out.append(path); break
            if op in FAIL or op not in EFFECT and not (0x80 <= op <= 0x9f):
                break
            if 0x80 <= op <= 0x8f:                                  # DUPn
                n = op - 0x7f
                if len(st) < n: break
                st.append(st[-n]); k += 1; continue
            if 0x90 <= op <= 0x9f:                                  # SWAPn
                n = op - 0x8f
                if len(st) < n + 1: break
                st[-1], st[-1 - n] = st[-1 - n], st[-1]; k += 1; continue
            pops, pushes = EFFECT[op]
            if len(st) < pops:                                      # stack underflow -> infeasible
                break
            args = [st.pop() for _ in range(pops)]
            if op == 0x56:                                          # JUMP
                t = args[0]
                if t is None or t not in c.jumpdests: break         # unresolvable target -> infeasible
                k = c.at[t]; continue
            if op == 0x57:                                          # JUMPI
                t, cond = args
                if t is None or t not in c.jumpdests: break
                if cond is not None and last is not None and visits.get(last, 0) > unroll:
                    cond = None                                     # bounded unrolling
                if cond is None:                                    # unknown: explore taken branch later
                    todo.append((c.at[t], tuple(st), tuple(path), dict(visits), last))
                    k += 1; continue
                k = c.at[t] if cond else k + 1; continue
            if 0x60 <= op <= 0x7f or op == 0x5f:
                st.append(v)
            elif op == 0x35:                                        # CALLDATALOAD
                st.append((selector << 224) if args[0] == 0 else None)
            elif op == 0x36:
                st.append(CALLDATASIZE)
            elif pushes:
                st.append(_alu(op, args))
            if len(st) > 1024: break
            k += 1
    return out


@lru_cache(maxsize=None)
def _fp_cached(cid, selector):
    return function_paths(_REG[cid], selector)


_REG = {}
def register(contract):
    _REG[id(contract)] = contract
    return id(contract)


def stitch(caller_path, callee, callee_id, max_stitches=8):
    """Splice callee function paths into a caller path at CALL-family opcodes, using the most recent PUSH4 constant
    since the previous call that is a selector the callee implements (DeFiTail-style signature matching; skips
    masks such as 0xffffffff). Returns (opcode names, n_stitches)."""
    seq, pending, n = [], [], 0
    for op, sel in caller_path:
        seq.append(NAMES.get(op, "UNKNOWN"))
        if sel is not None:
            pending.append(sel)
        if op in CALLS:
            hit = next((s for s in reversed(pending) if s in callee.selectors), None)
            if hit is not None and n < max_stitches:
                sub = _fp_cached(callee_id, hit)
                if sub:
                    seq.extend(NAMES.get(o, "UNKNOWN") for o, _ in sub[0])
                    n += 1
            pending = []
    return seq, n
E = sys.modules['__main__']   # helpers live in the notebook namespace; later cells call E.<name>

## 6. Compile the attacker contracts (exploit PoCs) with solc
No Foundry needed: `py-solc-x` + forge-std pinned to the repo's `foundry.lock` revision. Attacker side = every contract with code in
the PoC file. Results cached in `data/raw/attacker_bytecode/<incident>.json`.

In [8]:
FORGE_STD = DATA / "raw" / "DeFiHackLabs" / "lib" / "forge-std"
if not (FORGE_STD / "src").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/foundry-rs/forge-std", str(FORGE_STD)], check=True)
    subprocess.run(["git", "-C", str(FORGE_STD), "checkout", "-q", "3353993420c6e488a2914ce02d88174e80ad80f8"], check=True)


REPO = DATA / "raw" / "DeFiHackLabs"


def pick_version(src):
    m = re.search(r"pragma\s+solidity\s+([^;]+);", src)
    spec = m[1] if m else ">=0.8.0"
    installed = solcx.get_installed_solc_versions()
    try:
        return solcx.install_solc_pragma(f"pragma solidity {spec};") if not installed else _match(spec, installed)
    except Exception:
        return solcx.install_solc_pragma(f"pragma solidity {spec};")


def _match(spec, installed):
    from semantic_version import NpmSpec
    s = NpmSpec(spec.replace(" ", "") if "||" not in spec else spec)
    ok = [v for v in installed if s.match(v)]
    return max(ok) if ok else solcx.install_solc_pragma(f"pragma solidity {spec};")


def compile_poc(poc_rel):
    path = REPO / poc_rel
    src = path.read_text(encoding="utf-8", errors="ignore")
    ver = pick_version(src)
    rel = path.relative_to(REPO).as_posix()
    std = {
        "language": "Solidity",
        "sources": {rel: {"urls": [rel]}},
        "settings": {
            "remappings": ["forge-std/=lib/forge-std/src/"],
            "optimizer": {"enabled": True, "runs": 200},
            "outputSelection": {rel: {"*": ["evm.deployedBytecode.object", "abi"]}},
            **({"evmVersion": "shanghai"} if ver.minor == 8 and ver.micro >= 20 else {}),
        },
    }
    out = solcx.compile_standard(std, solc_version=ver, base_path=str(REPO), allow_paths=[str(REPO)])
    own = {}
    for name, c in out["contracts"].get(rel, {}).items():
        code = c["evm"]["deployedBytecode"]["object"]
        if code:  # interfaces / abstract contracts have no code
            own[name] = code
    main = next((n for n in own if re.search(rf"contract\s+{n}\s+is\s+[^{{]*\bTest\b", src)), None)
    return str(ver), main, own




D = DATA
OUT = D / "raw" / "attacker_bytecode"
OUT.mkdir(parents=True, exist_ok=True)


def job(args):
    iid, poc = args
    f = OUT / f"{iid}.json"
    if f.exists():
        return iid, json.loads(f.read_text())["status"]
    try:
        ver, main, own = compile_poc(poc)
        rec = dict(status="ok" if main else "no_main_contract", solc=ver, main=main, contracts=own)
    except Exception as e:
        rec = dict(status="error", error=str(e).strip().splitlines()[-1][:300] if str(e).strip() else repr(e))
    f.write_text(json.dumps(rec))
    return iid, rec["status"]



inc_c = pd.read_csv(DATA / "processed" / "dhl_incidents.csv", dtype=str).fillna("")
inc_c = inc_c[inc_c.bytecode_available == "1"]
from collections import Counter
print(Counter(job(a)[1] for a in zip(inc_c.incident_id, inc_c.poc)))   # cached per incident; ~40 min if uncached

Counter({'ok': 565, 'no_main_contract': 188, 'error': 12})


## 7. Resolve proxy victims to their implementation (EIP-1167 clones, EIP-1967 / EIP-1822 / legacy OZ / slots 0-2)

In [9]:
D = DATA
RPCS_PROXY = {"mainnet": "https://eth.drpc.org", "bsc": "https://bsc-mainnet.public.blastapi.io",
        "arbitrum": "https://arbitrum-one.public.blastapi.io", "base": "https://base.drpc.org",
        "polygon": "https://polygon.drpc.org", "optimism": "https://optimism.drpc.org",
        "avalanche": "https://api.avax.network/ext/bc/C/rpc"}
SLOTS = [("eip1967", "0x360894a13ba1a3210667c828492db98dca3e2076cc3735a920a3ca505d382bbc"),
         ("eip1822", "0xc5f16f0fcc639fa48a6947836d9850f504798523bf8c9a3a87d5876cf622bcf7"),
         ("oz_legacy", "0x7050c9e0f4ca769c69bd3a8ef740bc37934f8e2c036e5a723fd8ee048ed3f8c3"),
         ("slot0", "0x0"), ("slot1", "0x1"), ("slot2", "0x2")]
CLONE = re.compile(r"363d3d373d3d3d363d73([0-9a-f]{40})5af4")
sess = requests.Session()


def rpc(url, method, params):
    for attempt in range(6):
        try:
            r = sess.post(url, json={"jsonrpc": "2.0", "id": 1, "method": method, "params": params}, timeout=30).json()
            if "result" in r: return r["result"]
        except Exception:
            pass
        time.sleep(2 * (attempt + 1))
    return None


def resolve(job):
    chain, addr, block, sha = job
    url, outd = RPCS_PROXY[chain], D / "raw" / "bytecode" / chain
    code = (outd / f"{sha}.hex").read_text()
    cands = []
    m = CLONE.search(code)
    if m: cands.append(("eip1167", "0x" + m[1]))
    for name, slot in SLOTS:
        v = rpc(url, "eth_getStorageAt", [addr, slot, hex(block)])
        if v and int(v, 16) and int(v, 16) < (1 << 160):
            cands.append((name, "0x" + v[-40:]))
    for method, impl in cands:
        ic = rpc(url, "eth_getCode", [impl, hex(block)])
        if ic and len(ic) > 2:
            ic = ic[2:]
            c = E.Contract(ic)
            if len(c.selectors) >= 3:
                isha = hashlib.sha256(ic.encode()).hexdigest()[:16]
                if not (outd / f"{isha}.hex").exists(): (outd / f"{isha}.hex").write_text(ic)
                return chain, addr, block, impl.lower(), isha, method
    return chain, addr, block, "", "", "unresolved"


if True:
    inc = pd.read_csv(D / "processed" / "dhl_incidents.csv", dtype=str).fillna("")
    inc = inc[inc.bytecode_available == "1"]
    jobs = set()
    for r in inc.itertuples():
        for a, sha in zip(r.contracts.split(";"), r.code_shas.split(";")):
            if not a: continue
            c = E.Contract((D / "raw" / "bytecode" / r.chain / f"{sha}.hex").read_text())
            if any(op == 0xf4 for _, op, _ in c.ins) and len(c.selectors) < 4:
                jobs.add((r.chain, a, int(r.fork_block), sha))
    done = set()
    for f in (D / "raw" / "bytecode").glob("*/proxies.csv"):
        for x in csv.DictReader(open(f)): done.add((f.parent.name, x["address"], int(x["block"])))
    jobs = sorted(j for j in jobs if j[:3] not in done)
    print(len(jobs), "proxy-like contracts to resolve", flush=True)
    writers = {}
    with ThreadPoolExecutor(3) as ex:
        for i, (chain, addr, block, impl, isha, method) in enumerate(ex.map(resolve, jobs), 1):
            p = D / "raw" / "bytecode" / chain / "proxies.csv"
            new = not p.exists()
            with open(p, "a", newline="") as f:
                w = csv.writer(f)
                if new: w.writerow(["address", "block", "impl", "impl_sha", "method"])
                w.writerow([addr, block, impl, isha, method])
            if i % 50 == 0: print(i, flush=True)
    rows = []
    for f in (D / "raw" / "bytecode").glob("*/proxies.csv"):
        rows += list(csv.DictReader(open(f)))
    print("methods:", pd.DataFrame(rows).method.value_counts().to_dict())

0 proxy-like contracts to resolve


methods: {'eip1167': 165, 'eip1967': 114, 'unresolved': 87, 'slot0': 32, 'slot2': 15, 'slot1': 4}


## 8. Build pairwise data paths -> `data/processed/paths.csv`
For each incident and each **protocol** victim contract V (shared tokens/routers excluded): every attacker exploit-function path,
with only V stitched in (DeFiTail's pairwise view; other callees stay opaque). forge-std test helpers are excluded. Stack-only
opcodes (PUSH/DUP/SWAP/POP/JUMPDEST) are dropped to fit more semantics into the model window; max 3,000 tokens per path and
32 paths per incident (stitched paths first, spread over victims).

In [10]:
D = DATA
TEST_SELS = set(json.load(open(DATA / "processed" / "forge_std_test_selectors.json")))
DROP = {n for c, n in E.NAMES.items() if n.startswith(("PUSH", "DUP", "SWAP")) or n in ("POP", "JUMPDEST")}
MAX_PATHS_PER_INCIDENT = 32
MAX_TOKENS = 3000


def semantic(seq):
    return [t for t in seq if t not in DROP][:MAX_TOKENS]


def attacker_functions(att):
    """(contract, selector) pairs for the exploit logic: main contract minus forge-std helpers + all helper contracts."""
    cons = {n: E.Contract(code, n) for n, code in att["contracts"].items()}
    main = max(cons, key=lambda n: len(cons[n].selectors & TEST_SELS))
    if len(cons[main].selectors & TEST_SELS) < 10:
        return []
    fns = [(cons[main], s) for s in sorted(cons[main].selectors - TEST_SELS)]
    fns += [(c, s) for n, c in cons.items() if n != main for s in sorted(c.selectors)]
    return fns


def process(row):
    f = D / "raw" / "attacker_bytecode" / f"{row['incident_id']}.json"
    if not f.exists():
        return row["incident_id"], [], "no attacker bytecode"
    att = json.loads(f.read_text())
    if not att.get("contracts"):
        return row["incident_id"], [], "attacker compile failed"
    fns = attacker_functions(att)
    att_paths = [(c.name, s, p) for c, s in fns  # exploit functions are long call sequences -> larger limits
                 for p in E.function_paths(c, s, max_len=40000, max_steps=2000000)]
    if not att_paths:
        return row["incident_id"], [], "no feasible attacker path"
    victims = [(a, sha) for a, sha in zip(row["victims"].split(";"), row["victim_shas"].split(";")) if a]
    out = []
    for a, sha in victims:
        V = E.Contract((D / "raw" / "bytecode" / row["chain"] / f"{sha}.hex").read_text(), a)
        vid = E.register(V)
        for cname, s, p in att_paths:
            seq, n = E.stitch(p, V, vid)
            out.append(dict(pair_victim=a, attacker_fn=f"{cname}:{s:08x}", n_stitches=n, opcodes=" ".join(semantic(seq))))
    if not victims:  # every protocol contract is shared infra -> attacker-only paths
        for cname, s, p in att_paths:
            out.append(dict(pair_victim="", attacker_fn=f"{cname}:{s:08x}", n_stitches=0,
                            opcodes=" ".join(semantic([E.NAMES.get(o, "UNKNOWN") for o, _ in p]))))
    # cap: prefer stitched paths, spread over victims, deterministic
    out.sort(key=lambda r: (-min(r["n_stitches"], 1), hashlib.md5((r["pair_victim"] + r["attacker_fn"]).encode()).hexdigest()))
    by_v = defaultdict(list)
    for r in out: by_v[r["pair_victim"]].append(r)
    keep, i = [], 0
    while len(keep) < MAX_PATHS_PER_INCIDENT and any(by_v.values()):
        for v in list(by_v):
            if by_v[v] and len(keep) < MAX_PATHS_PER_INCIDENT: keep.append(by_v[v].pop(0))
    return row["incident_id"], keep, "ok"


if True:
    inc = pd.read_csv(D / "processed" / "dhl_incidents.csv", dtype=str).fillna("")
    inc = inc[inc.bytecode_available == "1"].copy()
        # shared infra (same rule as build_dataset.ipynb) -> victims = protocol contracts only
    full = pd.read_csv(D / "processed" / "dhl_incidents.csv", dtype=str).fillna("")
    uses = defaultdict(set)
    for r in full.itertuples():
        for a in filter(None, r.addresses.split(";")): uses[(r.chain, a)].add(r.incident_id)
    shared = {k for k, v in uses.items() if len(v) >= 5}
    # proxies -> implementation code at the same block (from resolve_proxies.py)
    impl = {}
    for f in (D / "raw" / "bytecode").glob("*/proxies.csv"):
        for x in pd.read_csv(f, dtype=str).fillna("").itertuples():
            if x.impl_sha: impl[(f.parent.name, x.address, x.block)] = x.impl_sha
    rows = []
    for r in inc.itertuples():
        pairs = [(a, impl.get((r.chain, a, r.fork_block), s)) for a, s in zip(r.contracts.split(";"), r.code_shas.split(";"))
                 if a and (r.chain, a) not in shared]
        rows.append(dict(incident_id=r.incident_id, chain=r.chain, victims=";".join(a for a, _ in pairs),
                         victim_shas=";".join(s for _, s in pairs)))
    meta = inc.set_index("incident_id")
    recs, status = [], {}
    if True:
        for i, (iid, paths, st) in enumerate(map(process, rows), 1):
            status[iid] = st
            m = meta.loc[iid]
            for p in paths:
                recs.append(dict(event_id=iid, chain=m.chain, n_contracts=int(m.n_contracts), bucket=m.bucket,
                                 access_control=int(m.label_access_control), flash_loan=int(m.label_flash_loan_type),
                                 vuln_type=m.vuln_type, **p))
            if i % 50 == 0: print(i, "incidents,", len(recs), "paths", flush=True)
    df = pd.DataFrame(recs)
    df.insert(0, "id", range(len(df)))
    name = "paths.csv"
    df.to_csv(D / "processed" / name, index=False)
    pd.Series(status, name="status").to_csv(D / "processed" / "path_build_status.csv")
    print("status:", pd.Series(status).value_counts().to_dict())
    print("paths:", len(df), "| incidents with paths:", df.event_id.nunique())
    if len(df):
        print("stitched share:", round((df.n_stitches > 0).mean(), 3), "| tokens per path median:", int(df.opcodes.str.count(" ").median()) + 1)

50 incidents, 624 paths


100 incidents, 1283 paths


150 incidents, 1965 paths


200 incidents, 2700 paths


250 incidents, 3494 paths


...

650 incidents, 8537 paths


700 incidents, 9196 paths


750 incidents, 9999 paths


status: {'ok': 704, 'no feasible attacker path': 49, 'attacker compile failed': 12}
paths: 10248 | incidents with paths: 704


stitched share: 0.5 | tokens per path median: 851
